# 5 - Augmentation de Texte avec Bruit OCR Réaliste

## Objectif
Générer des variantes **bruitées de façon réaliste** pour les catégories sous-représentées.

Les patterns de bruit sont extraits directement des **vraies erreurs EasyOCR** observées dans `textes_bruts_tickets.csv`.

## Pipeline
```
Produit propre (ex: 'Shampoing Elsève')
    ↓  Simulateur de bruit OCR
Texte bruité (ex: '5hemp0ing EI5ève') 
    ↓  Ajout dans echantillon_annote.csv
Modèle plus robuste en production ✅
```

## Catégories cibles
| Catégorie | Exemples actuels | Objectif |
|-----------|-----------------|----------|
| Entretien | ~8 | 80+ |
| Hygiène_et_Beauté | ~10 | 80+ |
| Divers/Bazar | ~15 | 80+ |
| Restauration | ~15 | 80+ |
| Boissons | ~50 | 120+ |

In [ ]:
# --- IMPORTS ---
import pandas as pd
import random
import re
import numpy as np

# Graine pour la reproductibilité
random.seed(42)
np.random.seed(42)

print('✅ Imports OK')

## 1. Analyse des vrais patterns d'erreurs OCR
Ces patterns sont **extraits manuellement** en comparant le texte OCR brut avec les annotations. Ils reflètent les vraies erreurs d'EasyOCR sur des photos de tickets de caisse.

In [ ]:
# =============================================================================
# PATTERNS D'ERREURS OCR — Extraits des vraies données (textes_bruts_tickets.csv)
# Chaque pattern est observé dans les fichiers du dataset réel
# =============================================================================

# --- Pattern 1 : Confusion chiffre <-> lettre ---
# Observé : '1ombtedarticles' (l→1), '5OCL' (S→5), '[AIT' (L→[), 'K1NDER' (I→1)
CONFUSION_CHIFFRE_LETTRE = {
    'l': ['1', 'I', '|'],
    'L': ['1', '[', 'I'],
    'I': ['1', 'l', '|'],
    'i': ['1', '!', 'l'],
    'S': ['5', '$'],
    's': ['5', '$'],
    'O': ['0', 'Q'],
    'o': ['0', 'Q'],
    'G': ['6', 'C'],
    'g': ['9', 'q'],
    'B': ['8', '6'],
    'b': ['6', 'd'],
    'Z': ['2', '7'],
    'z': ['2'],
    'T': ['7', '+'],
    'e': ['3'],
    'A': ['4', '@'],
    'a': ['4', '@'],
    'q': ['9', 'g'],
    'D': ['0', 'Q'],
}

# --- Pattern 2 : Majuscule/minuscule aléatoire ---
# Observé : 'MiLKA', 'CoMP', 'innoCENT', 'GBAPuPOMME', 'TABDULE', 'GaLETTE'
# Parfois une lettre sur deux change, parfois juste une lettre en début de mot

# --- Pattern 3 : Caractères parasites insérés ---
# Observé : 'SAND .', 'PQ.8', 'BOUL /VDE', 'CoMP _', 'GBAPu', 'BML7b:'
CHARS_PARASITES = ['.', ',', '_', '/', ':', '#', '*', '+', "'", '!', '-', '?', '&']

# --- Pattern 4 : Lettres dupliquées ---
# Observé : 'Coecthhrket' (e → ee, h → hh), 'TRAITEMENT' peut devenir 'TRAIITEMENT'

# --- Pattern 5 : Lettre manquante ---
# Observé : 'SCHTROUMP' (SCHTROUMPF), 'CEREAL' (CEREALE), 'POUB' (POUBELLE)

# --- Pattern 6 : Accents déformés ---
# Observé : 'Oté Dêsignatien', 'MADELÈI', 'ücocciMarket'
ACCENTS_ERRONES = {
    'e': ['ë', 'ê', 'é', 'è', '3'],
    'a': ['à', 'â', 'ä', '@'],
    'i': ['ï', 'î', '1'],
    'u': ['ü', 'û', 'ù'],
    'o': ['ô', 'ö', '0'],
    'c': ['ç'],
    'E': ['Ë', 'Ê', 'É', 'È'],
    'A': ['À', 'Â', 'Ä'],
}

# --- Pattern 7 : Mots fusionnés ou coupés ---
# Observé : 'CANTHEINEKEN', 'squthenEUR', '1ombtedarticles'
# => fusion de deux mots adjacents

# --- Pattern 8 : Troncature en fin de ligne ---
# Observé : 'INNOCENT TUTTI FRU' (truncated), 'CHIPS SAV POMME FR'

# --- Pattern 9 : Confiance OCR faible = texte presque incompréhensible ---
# Observé : 'L{biuF TrOCOLZT Pu', 'DMUIL {L7Sîv', 'oreclhIEITE'
# On simule ça en remplaçant plus de caractères

print('✅ Patterns OCR définis')
print(f'   - {len(CONFUSION_CHIFFRE_LETTRE)} confusions chiffre/lettre')
print(f'   - {len(CHARS_PARASITES)} caractères parasites')
print(f'   - {len(ACCENTS_ERRONES)} substitutions accent')

## 2. Moteur de bruit OCR

In [ ]:
def appliquer_bruit_ocr(texte, niveau='moyen'):
    """
    Applique un bruit OCR réaliste sur un texte propre.
    
    Niveaux :
    - 'faible'  : confiance OCR haute (0.7-1.0) — quelques erreurs
    - 'moyen'   : confiance OCR moyenne (0.3-0.7) — erreurs typiques
    - 'fort'    : confiance OCR basse (0.0-0.3) — texte très dégradé
    """
    if not isinstance(texte, str) or len(texte) == 0:
        return texte
    
    # Probabilités selon le niveau
    probs = {
        'faible': {
            'confusion_char':   0.04,  # 4% des caractères
            'casse_aleatoire':  0.05,  # 5% des lettres changent de casse
            'parasite':         0.05,  # 5% de chance d'ajouter un parasite
            'duplication':      0.02,  # 2% des lettres sont doublées
            'suppression':      0.02,  # 2% des lettres sont supprimées
            'accent_errone':    0.03,  # 3% des voyelles ont un accent erroné
            'troncature':       0.10,  # 10% de chance de tronquer la fin
            'fusion':           0.05,  # 5% de chance de fusionner deux mots
        },
        'moyen': {
            'confusion_char':   0.10,
            'casse_aleatoire':  0.15,
            'parasite':         0.15,
            'duplication':      0.05,
            'suppression':      0.05,
            'accent_errone':    0.08,
            'troncature':       0.20,
            'fusion':           0.10,
        },
        'fort': {
            'confusion_char':   0.25,
            'casse_aleatoire':  0.30,
            'parasite':         0.25,
            'duplication':      0.10,
            'suppression':      0.12,
            'accent_errone':    0.15,
            'troncature':       0.35,
            'fusion':           0.20,
        }
    }
    p = probs[niveau]
    
    resultat = list(texte)
    
    # --- 1. Confusion chiffre/lettre ---
    for i, c in enumerate(resultat):
        if c in CONFUSION_CHIFFRE_LETTRE and random.random() < p['confusion_char']:
            resultat[i] = random.choice(CONFUSION_CHIFFRE_LETTRE[c])
    
    # --- 2. Casse aléatoire ---
    for i, c in enumerate(resultat):
        if c.isalpha() and random.random() < p['casse_aleatoire']:
            resultat[i] = c.lower() if c.isupper() else c.upper()
    
    # --- 3. Accents erronés ---
    for i, c in enumerate(resultat):
        if c in ACCENTS_ERRONES and random.random() < p['accent_errone']:
            resultat[i] = random.choice(ACCENTS_ERRONES[c])
    
    # --- 4. Duplication de lettre ---
    i = 0
    while i < len(resultat):
        if resultat[i].isalpha() and random.random() < p['duplication']:
            resultat.insert(i + 1, resultat[i])
            i += 2  # Sauter la copie
        else:
            i += 1
    
    # --- 5. Suppression de lettre ---
    resultat_filtré = []
    for c in resultat:
        if c.isalpha() and random.random() < p['suppression']:
            pass  # On supprime le caractère
        else:
            resultat_filtré.append(c)
    resultat = resultat_filtré
    
    # Reconvertir en texte pour les traitements au niveau des mots
    texte_interim = ''.join(resultat)
    
    # --- 6. Insertion de caractère parasite ---
    if random.random() < p['parasite']:
        mots = texte_interim.split()
        if len(mots) >= 1:
            pos = random.randint(0, len(mots))
            mots.insert(pos, random.choice(CHARS_PARASITES))
            texte_interim = ' '.join(mots)
    
    # --- 7. Fusion de mots adjacents ---
    if random.random() < p['fusion']:
        mots = texte_interim.split()
        if len(mots) >= 2:
            idx = random.randint(0, len(mots) - 2)
            mots[idx] = mots[idx] + mots[idx + 1]
            del mots[idx + 1]
            texte_interim = ' '.join(mots)
    
    # --- 8. Troncature de fin de texte ---
    if random.random() < p['troncature'] and len(texte_interim) > 6:
        # On coupe les N derniers caractères (simule une ligne trop longue)
        couper = random.randint(2, max(3, len(texte_interim) // 4))
        texte_interim = texte_interim[:-couper]
    
    # Sécurité : ne jamais retourner une chaîne vide
    if len(texte_interim.strip()) < 2:
        return texte
    
    return texte_interim.strip()


# === TEST RAPIDE ===
print('🧪 TEST DU MOTEUR DE BRUIT OCR')
print('='*50)

exemples_test = [
    'SHAMPOING ELSÈVE LISSE',
    'DÉTERGENT LIQUIDE VAISSELLE',
    'COCA COLA 1.5L',
    'TARTE CITRON MERINGUEE',
    'COTON TIGE 300 PIECES',
]

for texte in exemples_test:
    print(f'\nOriginal  : {texte}')
    print(f'Faible    : {appliquer_bruit_ocr(texte, "faible")}')
    print(f'Moyen     : {appliquer_bruit_ocr(texte, "moyen")}')
    print(f'Fort      : {appliquer_bruit_ocr(texte, "fort")}')

## 3. Banque de produits propres par catégorie

Ces produits sont des **noms réels** de produits de supermarché français, adaptés aux catégories sous-représentées. Ils seront bruités pour simuler des erreurs OCR réalistes.

In [ ]:
# =============================================================================
# BANQUE DE PRODUITS PROPRES PAR CATÉGORIE
# Format : noms courts comme sur les vrais tickets de caisse
# =============================================================================

PRODUITS = {

    # =========================================================================
    # ENTRETIEN
    # =========================================================================
    'Entretien': [
        # Lessive
        'LESSIVE LIQUIDE ARIEL',
        'LESSIVE POUDRE SKIP',
        'LESSIVE CAPSULES LENOR',
        'GEL LESSIVE PERSIL',
        'LESSIVE FRAICHEUR DASH',
        'LESSIVE COLOR ARIEL',
        'PASTILLES LESSIVE SKIP',
        'LESSIVE BEBE AMBRE SOLAIRE',
        # Nettoyants
        'LIQUIDE VAISSELLE FAIRY',
        'LIQUIDE VAISSELLE PAIC',
        'TABLETTES LAVE VAISSELLE FINISH',
        'PRODUIT SOL AJAX',
        'NETTOYANT MENAGER MR PROPRE',
        'SPRAY MULTI SURFACES FLASH',
        'WC NET CANARD',
        'NETTOYANT SALLE DE BAIN',
        'JAVEL LIQUIDE ST MARC',
        'NETTOYANT VITRE GLASSS',
        'LINGETTES NETTOYANTES EASYWIPES',
        'DEGRAISSANT CUISINE AJAX',
        # Papier
        'ESSUIE TOUT BOUNTY',
        'PAPIER TOILETTE LOTUS',
        'PAPIER WC DOUBLE RENOVA',
        'MOUCHOIRS KLEENEX',
        'SACS POUBELLE 50L',
        'SACS CONGELATION ZIP',
        'FILM ETIRABLE TOPPITS',
        'PAPIER ALUMINIUM ALBAL',
        # Divers entretien
        'EPONGE GRATTANTE SCOTCH',
        'GANTS MENAGE MAPA',
        'DESODORISANT AIR AMBI PUR',
        'BOUGIES DEVILISS',
        'ANTI CALCAIRE RUBIGINE',
        'DESODOISANT FEBREZE',
        'SOPALIN FANTASTIQUE',
        'LIQUIDE VAISSELLE CITRON',
        'NETTOYANT SOL PARQUET',
        'GEL DOUCHE DESINFECTANT',
    ],

    # =========================================================================
    # HYGIÈNE ET BEAUTÉ
    # =========================================================================
    'Hygiène_et_Beauté': [
        # Soins cheveux
        'SHAMPOING ELSEVE LISSE',
        'SHAMPOING PANTENE PRO',
        'APRES SHAMPOING HERBAL',
        'SHAMPOING HEAD SHOULDERS',
        'MASQUE CHEVEUX GARNIER',
        'HUILE CHEVEUX MOROCCANOIL',
        'LAQUE 300 FORTE ELNETT',
        'GEL COIFFANT STUDIO LINE',
        # Soins corps
        'GEL DOUCHE DOVE HYDRA',
        'GEL DOUCHE NIVEA HOMME',
        'SAVON LIQUIDE SANEX',
        'CREME HYDRATANTE NIVEA',
        'LAIT CORPS DOVE',
        'DEODORANT DOVE ORIGINAL',
        'DEO SPRAY REXONA',
        'DEODORANT BILLE GARNIER',
        'DEODORANT NARTA HOMME',
        'BRUT ATO DEO ORIGINAL',
        # Soins visage
        'CREME VISAGE OLAY',
        'MOUSSE A RASER GILLETTE',
        'RASOIR GILLETTE MACH3',
        'LAMES RASOIR WILKINSON',
        'FOND DE TEINT GEMEY',
        'ROUGE A LEVRES LOREAL',
        'MASCARA LOREAL PARIS',
        'DEMAQUILLANT YEUX BIODERMA',
        # Hygiène dentaire
        'DENTIFRICE COLGATE TOTAL',
        'BROSSE A DENTS ORAL B',
        'BAIN DE BOUCHE LISTERINE',
        'FIL DENTAIRE ORAL B',
        # Autre hygiène
        'COTON TIGE 300 PIECES',
        'COTON DEMAQUILLANT',
        'COUCHES PAMPERS T3',
        'COUCHES XL HUGGIES',
        'PROTECTIONS PERIODIQUES ALWAYS',
        'SERVIETTES HYGIENES NANA',
        'PARFUM EAU TOILETTE AZZARO',
        'CREME SOLAIRE SPF50',
        'APRES SOLEIL NIVEA',
        'SERUM VISAGE LOREAL',
    ],

    # =========================================================================
    # DIVERS / BAZAR
    # =========================================================================
    'Divers/Bazar': [
        # Papeterie
        'STYLOS BILLE BIC BLEU',
        'CAHIER 96 PAGES SEYES',
        'POST IT 75X75 JAUNE',
        'CLASSEUR A4 OXFORD',
        'FEUTRES STABILO BOSS',
        'ENVELOPPES C5 BLANCHES',
        'CHEMISES CARTON ROUGE',
        'TROMBONE BOITE 100',
        'COLLE STICK UHU',
        'CISEAUX BUREAU',
        # Cuisine / Maison
        'SAC CABAS REUTILISABLE',
        'BOITE ALIMENTAIRE 2.2L',
        'BOITE HERMETIQUE 1L',
        'THERMOS INOX 500ML',
        'ASSIETTES CARTON X10',
        'GOBELETS PLASTIQUE X20',
        'PAILLE PAPIER X50',
        # Jardinage
        'RATEAU FEUILLE 24 DENTS',
        'GANTS JARDINAGE',
        'POTS FLEURS TERRACOTTA',
        'TERREAU UNIVERSEL 20L',
        'ENGRAIS PLANTES',
        # Bricolage
        'RUBAN ADHESIF SCOTCH',
        'AMPOULE LED E27 9W',
        'PILE AAA DURACELL',
        'PILE AA ENERGIZER X8',
        # Accessoires
        'PARAPLUIE AUTOMATIQUE',
        'LUNETTES SOLEIL',
        'BONNET HIVER',
        'CHAUSSETTES COTON X3',
        'TAPIS ANTIDERAPANT',
        'CINTRE METAL X10',
        # Bébé / Enfants
        'JOUET EVEIL BEBE',
        'LIVRE ENFANT IMAGIER',
        'CRAYONS COULEUR X12',
        'PEINTURE GOUACHE X6',
        'SET 120 FEUTRES',
        'SAC ISOTHERME REPAS',
        'MUG ENJOY DECORE',
    ],

    # =========================================================================
    # RESTAURATION (cafés, restaurants, snacks avec service)
    # =========================================================================
    'Restauration': [
        # Plats chauds
        'BAVETTE FRITES MAISON',
        'ENTRECOTE FRITES',
        'POULET ROTI JANZE',
        'POULET BASQUAISE',
        'GRATIN DAUPHINOIS',
        'BOEUF BOURGUIGNON',
        'BLANQUETTE VEAU',
        'STEAK HACHE FRITES',
        'BROCHETTE BOEUF',
        'AIGUILLETTES DE POULET',
        'FILET DE LIEU BEURRE',
        'HUITRES CHAUDES',
        'PLATEAU DE FRUITS MER',
        # Entrées / Salades
        'SALADE CESAR COMPOSEE',
        'MELI MELO CRUDITES',
        'COCKTAIL JUS FRUITS',
        'SOUPE DU JOUR',
        'VELOUTE POTIRON',
        # Desserts
        'TARTE CITRON MERINGUEE',
        'ILE FLOTTANTE GOURMANDE',
        'PROFITEROLES CARAMEL',
        'BROWNIE CHOCOLAT',
        # Pains / Sandwichs
        'PAIN CLASSIQUE',
        'BAGEL SAUMON CREAM',
        'BURGER FROMAGE FONDU',
        'CHEESEBURGER',
        'HOT DOG MOUTARDE',
        'PANINI JAMBON FROMAGE',
        # Boissons resto
        'CAFE EXPRESS',
        'CAFE DOUBLE EXPRESSO',
        'THE VERT JASMIN',
        'DEMI PRESSION BLONDE',
        'VIN ROUGE VERRE',
        'VIN BLANC PICHET',
        'ORANGINA GLASS',
        'ICE TEA PECHE RESTO',
        'EAU PLATE CARAFE',
        'LIMONADE MAISON',
    ],

    # =========================================================================
    # BOISSONS (supermarché)
    # =========================================================================
    'Boissons': [
        # Bières
        'HEINEKEN 5OCL CANETTE',
        'CORONA EXTRA 33CL',
        'LEFFE BLONDE 75CL',
        'LEFFE BRUNE 75CL',
        'GRIMBERGEN TRIPEL 75CL',
        'CHOUFFE 75CL BLONDE',
        'DESPERADOS 5OCL BIERE',
        'PACK 6 BIERE 33CL',
        'BIERE 3 MONTS 5OCL',
        'GINGER BEER 5OCL',
        'CH TI BLONDE 5OCL BTE',
        'HOEGAARDEN 5OCL BIERE',
        # Eaux
        'EVIAN 1.5L PET',
        'VOLVIC 1L BOUTEILLE',
        'BADOIT ROUGE NATUR',
        'SAN PELLEGRINO 75CL',
        'PERRIER 33CL BOUTEILLE',
        'GRANDE FONTAINE 6X1.5L',
        'CRISTALLINE 6X1.5L',
        'VITTEL SPORT CAP',
        # Sodas
        'COCA COLA 1.5L PET',
        'PEPSI MAX 1.5L',
        'FANTA ORANGE 1L',
        'SPRITE 1.5L',
        'ORANGINA 1.5L',
        'SCHWEPPES TONIC 1L',
        'LIPTON ICE TEA PECHE',
        'OASIS TROPICAL 1.5L',
        # Jus
        'JUS ORANGE TROPICANA 1L',
        'INNOCENT TUTTI FRUTTI',
        'CAPRI SUN MULTIVIT',
        'JUS POMME AUCHAN',
        'NECTAR PECHE ANDROS',
        # Alcools / Vins
        'VIN ROUGE BDX 75CL',
        'ROSE PROVENCE 75CL',
        'CHABLIS BLANC 75CL',
        'CHAMPAGNE NICOLAS 75CL',
        'CIDRE BRETON 75CL',
        'PASTIS RICARD 70CL',
        'WHISKY BALLANTINE 70CL',
        # Boissons chaudes
        'CAFE NESPRESSO X10',
        'THE EARL GREY TWININGS',
        'CHOCOLAT CHAUD VAN HOUTEN',
        'CHICHOREE RICOFFY',
    ],
}

# Bilan
print('📦 Banque de produits chargée :')
total = 0
for cat, produits in PRODUITS.items():
    print(f'   {cat:25s}: {len(produits)} produits')
    total += len(produits)
print(f'   {"":25s}  -------')
print(f'   {"TOTAL":25s}: {total} produits')

## 4. Génération des données bruitées

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
VARIANTES_PAR_PRODUIT = 15   # Nombre de variantes bruitées par produit propre
                              # Avec ~40 produits/catégorie × 15 = ~600 exemples/catégorie

# Distribution des niveaux de bruit (reproduit la réalité :
# beaucoup d'images ok, quelques-unes très dégradées)
DISTRIBUTION_NIVEAUX = [
    ('faible', 0.35),   # 35% — OCR confiance haute
    ('moyen',  0.45),   # 45% — OCR confiance moyenne
    ('fort',   0.20),   # 20% — OCR confiance faible
]

# Valeurs fictives mais réalistes pour les colonnes du CSV
NOM_FICHIER_FAKE = 'fake_augmented'

# =============================================================================
# GÉNÉRATION
# =============================================================================
def choisir_niveau():
    """Choisit un niveau de bruit selon la distribution réelle observée."""
    r = random.random()
    cumul = 0
    for niveau, prob in DISTRIBUTION_NIVEAUX:
        cumul += prob
        if r < cumul:
            return niveau
    return 'moyen'

lignes_generees = []

print('🏭 Génération des données bruitées...')
print('='*60)

for categorie, produits in PRODUITS.items():
    nb_avant = len(lignes_generees)
    
    for produit_propre in produits:
        for i in range(VARIANTES_PAR_PRODUIT):
            niveau = choisir_niveau()
            texte_bruite = appliquer_bruit_ocr(produit_propre, niveau)
            
            lignes_generees.append({
                'nom_fichier':      f'{NOM_FICHIER_FAKE}_{categorie[:4].lower()}_{i}',
                'texte_brut':       texte_bruite,
                'y_centre':         round(random.uniform(700, 1200), 1),
                'x_gauche':         round(random.uniform(250, 700), 1),
                'est_un_prix':      False,
                'confiance_ocr':    round(random.uniform(0.02, 0.99), 2),
                'categorie_cible':  categorie,
                'texte_original':   produit_propre,   # Colonne bonus pour debug
                'niveau_bruit':     niveau,            # Colonne bonus pour debug
            })
    
    nb_apres = len(lignes_generees)
    print(f'  ✅ {categorie:25s} : +{nb_apres - nb_avant} exemples')

df_fake = pd.DataFrame(lignes_generees)

print()
print(f'📊 TOTAL GÉNÉRÉ : {len(df_fake)} lignes')
print(f'   Répartition par catégorie :')
print(df_fake['categorie_cible'].value_counts().to_string())

## 5. Vérification qualité
Vérifions que le bruit ressemble bien aux vraies erreurs OCR de notre dataset.

In [ ]:
print('🔍 EXEMPLES PAR CATÉGORIE — Comparaison original vs bruité')
print('='*70)

for categorie in PRODUITS.keys():
    print(f'\n📁 {categorie}')
    print(f'   {"Texte original":<35} {"Texte bruité":<35} Niveau')
    print(f'   {"-"*35} {"-"*35} ------')
    
    echantillon = df_fake[df_fake['categorie_cible'] == categorie].sample(
        n=min(5, len(df_fake[df_fake['categorie_cible'] == categorie])),
        random_state=42
    )
    
    for _, row in echantillon.iterrows():
        orig = row['texte_original'][:34]
        brut = row['texte_brut'][:34]
        niv  = row['niveau_bruit']
        print(f'   {orig:<35} {brut:<35} {niv}')

In [ ]:
# Vérification : distribution des niveaux de bruit
print('📈 Distribution des niveaux de bruit générés :')
print(df_fake['niveau_bruit'].value_counts())
print()

# Vérification : longueur moyenne des textes bruités vs originaux
avg_orig = df_fake['texte_original'].str.len().mean()
avg_brut = df_fake['texte_brut'].str.len().mean()
print(f'📏 Longueur moyenne :')
print(f'   Originaux  : {avg_orig:.1f} caractères')
print(f'   Bruités    : {avg_brut:.1f} caractères (-{avg_orig - avg_brut:.1f} en moyenne → troncatures OK)')
print()

# Vérification : aucun texte vide
vides = (df_fake['texte_brut'].str.strip() == '').sum()
print(f'⚠️  Textes vides après bruitage : {vides} (doit être 0)')

## 6. Fusion avec le dataset existant et sauvegarde

In [ ]:
# =============================================================================
# CONFIGURATION SAUVEGARDE
# =============================================================================
FICHIER_ANNOTE_EXISTANT    = 'echantillon_annote.csv'
FICHIER_FAKE_SEUL          = 'donnees_fake_bruitees.csv'   # Données fake seules
FICHIER_MERGE_COMPLET      = 'echantillon_annote_enrichi.csv'  # Fusion complète

# Colonnes compatibles avec echantillon_annote.csv (sans les colonnes de debug)
COLONNES_CSV = ['nom_fichier', 'texte_brut', 'y_centre', 'x_gauche',
                'est_un_prix', 'confiance_ocr', 'categorie_cible']

# --- 1. Sauvegarde des données fake seules ---
df_fake_export = df_fake[COLONNES_CSV].copy()
df_fake_export.to_csv(FICHIER_FAKE_SEUL, index=False, encoding='utf-8')
print(f'💾 Données fake sauvegardées : {FICHIER_FAKE_SEUL} ({len(df_fake_export)} lignes)')

# --- 2. Chargement du dataset existant ---
try:
    df_existant = pd.read_csv(FICHIER_ANNOTE_EXISTANT)
    # On ne garde que les lignes ANNOTÉES (avec categorie_cible)
    df_existant_propre = df_existant.dropna(subset=['categorie_cible']).copy()
    print(f'📂 Dataset existant chargé : {len(df_existant_propre)} lignes annotées')
except FileNotFoundError:
    print(f'⚠️  Fichier {FICHIER_ANNOTE_EXISTANT} introuvable. Uniquement les données fake seront sauvegardées.')
    df_existant_propre = pd.DataFrame(columns=COLONNES_CSV)

# --- 3. Fusion ---
df_merge = pd.concat([df_existant_propre[COLONNES_CSV], df_fake_export], 
                      ignore_index=True)
df_merge = df_merge.sample(frac=1, random_state=42).reset_index(drop=True)  # Mélange

# --- 4. Sauvegarde du merged ---
df_merge.to_csv(FICHIER_MERGE_COMPLET, index=False, encoding='utf-8')

print()
print('='*60)
print('✅ RÉSUMÉ FINAL')
print('='*60)
print(f'Dataset existant : {len(df_existant_propre):>6} lignes')
print(f'Données fake     : {len(df_fake_export):>6} lignes')
print(f'Dataset enrichi  : {len(df_merge):>6} lignes')
print()
print('📊 Distribution finale par catégorie :')
print(df_merge['categorie_cible'].value_counts().to_string())
print()
print(f'💾 Fichier prêt pour l\'entraînement : {FICHIER_MERGE_COMPLET}')

## 7. Lancement de l'entraînement
Une fois ce notebook exécuté, modifie `4_entrainement_ia.ipynb` pour pointer vers le nouveau fichier :

```python
# Dans 4_entrainement_ia.ipynb, changer :
FICHIER_ANNOTÉ = 'echantillon_annote.csv'
# Par :
FICHIER_ANNOTÉ = 'echantillon_annote_enrichi.csv'
```

Et ajouter `stratify=y` pour garantir une distribution équilibrée dans le test set :

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # ← AJOUTER stratify
)
```